<h2>Middleware</h2>


Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 

Tracking agent behavior with logging, analytics, and debugging.

Transforming prompts, tool selection, and output formatting.

Adding retries, fallbacks, and early termination logic.

Applying rate limits, guardrails, and PII detection.

In [1]:
import os

In [48]:
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

Summarization middleware: Automatically summarizes conversation history when apporaching token limits, preserving recet messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.

2. Milti-turn dialogues with extensive history.

3. Applications where preserving full conversation context matters.

In [49]:
from langchain.agents import  create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

In [50]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

In [51]:
llm = init_chat_model("groq:llama-3.3-70b-versatile")

In [52]:
agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [53]:
###Run with thread id
config={"configurable":{"thread_id":"test_thread"}}

In [54]:
questions=[
    'What is the capital of France?',
    'What is the capital of Germany?',
    "What is 2+2?",
    "What is the largest planet in our solar system?",
    "What is the boiling point of water?",
    "What is the speed of light?"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(f"Message:{response}")
    print(f"Messages: {len(response['messages'])}")

Message:{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='586db906-650f-4a85-9e16-032c9c4037f9'), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.010928896, 'completion_tokens_details': None, 'prompt_time': 0.002487518, 'prompt_tokens_details': None, 'queue_time': 0.055853609, 'total_time': 0.013416414}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e186c-35ee-7821-aa38-acff2461d109-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}
Messages: 2
Message:{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={

Token Size

In [55]:
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotls in {city}:
    1. Grand Hotel - 5 star, $359/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, buisiness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [56]:
agent1=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test_thread_2"}}

def count_tokens(messages):
    total_tokens=sum(len(str(m.content)) for m in messages)
    return total_tokens//4

In [57]:
#Run test
cities=['Paris','London','Tokyo','New York','Sydney','Dubai']

for city in cities:
    response=agent.invoke(
        {'messages':[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens=count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~761 tokens, 2 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='21007ac9-5c83-4bd3-9232-144c547b809e'), AIMessage(content="The City of Light. Paris, the capital of France, is famous for its stunning architecture, art museums, fashion, and romantic atmosphere. Here are some top hotels in Paris, categorized by price range:\n\n**Luxury Hotels**\n\n1. **Shangri-La Hotel, Paris** - A 5-star hotel with elegant rooms and a stunning view of the Eiffel Tower. (Rates start at around €600/night)\n2. **Four Seasons Hotel George V Paris** - A 5-star hotel with luxurious rooms, a spa, and a Michelin-starred restaurant. (Rates start at around €800/night)\n3. **The Ritz Paris** - A 5-star hotel with lavish rooms, a world-class spa, and a famous bar. (Rates start at around €1,000/night)\n4. **La Réserve Hotel and Spa** - A 5-star hotel with elegant rooms, a spa, and a Michelin-starred restaurant. (Rates start at around €700/night)\n5. **Hôtel

Fraction

In [58]:
agent1=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005), #0.5% = ~640 tokens
            keep=("fraction", 0.002) #0.2% = ~256 tokens
        )
    ]
)

Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).

Compliance workflows where human oversight is mandatory.

Long-running conversations where human feedback quides the agent.

In [74]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject}"

In [81]:
Agent=create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [82]:
config={"configurable":{"thread_id":"email_thread"}}

result=Agent.invoke(
    {"messages":[HumanMessage(content="Send an email to John@test.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am'")]},
    config=config
)

In [83]:
result

{'messages': [HumanMessage(content="Send an email to John@test.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am'", additional_kwargs={}, response_metadata={}, id='e1244069-71a0-4319-a2b1-999fbbdc0c3b'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'n34rd48kk', 'function': {'arguments': '{"body":"Let\'s meet tomorrow at 10am","recipient":"John@test.com","subject":"Meeting"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 315, 'total_tokens': 352, 'completion_time': 0.060681482, 'completion_tokens_details': None, 'prompt_time': 0.016167472, 'prompt_tokens_details': None, 'queue_time': 0.055546283, 'total_time': 0.076848954}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1871-d63c-7c00-ab00-536e6844c59a-0', tool_calls=[{

In [84]:
from langgraph.types import Command

In [85]:
configuration={"configurable":{"thread_id":"email_thread"}}

In [86]:
if "__interrupt__" in result:
    print("Paused! Approving...")

    result=Agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type":"approve"
                    }
                ]
            }
        ),
        config=configuration
    )

Paused! Approving...


In [87]:
print(result['messages'][-1].content)

The email has been sent to John@test.com with the subject 'Meeting' and the body 'Let's meet tomorrow at 10am'.
